In [ ]:
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-distutils python3.10-venv -y

# register python versions
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 2
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.12 1

# set python3 = python3.10 (NO INTERACTIVE PROMPT)
!sudo update-alternatives --set python3 /usr/bin/python3.10

# install pip for python3.10
!curl -sS https://bootstrap.pypa.io/get-pip.py | sudo python3.10

!python3 --version
!pip3 --version


In [ ]:
!pip install -r "/kaggle/input/requirements/requirements.txt"



In [ ]:

!pip install protobuf==3.20.3


In [ ]:
import os
import shutil

paths_to_delete = [
    "/kaggle/working/tomato/"
]

for path in paths_to_delete:
    if os.path.exists(path):
        if os.path.isfile(path):
            os.remove(path)
            print("Deleted file:", path)
        elif os.path.isdir(path):
            shutil.rmtree(path)
            print("Deleted folder:", path)
    else:
        print("Not found:", path)


In [ ]:
import os

dataset_path = "/kaggle/working/balanced_data/"  # ⬅️ change to your dataset folder

folders = os.listdir(dataset_path)

for folder in folders:
    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):
        count = len([f for f in os.listdir(folder_path)
                     if os.path.isfile(os.path.join(folder_path, f))])

        print(f"{folder}: {count} images")


In [ ]:
import shutil
import os

src = "/kaggle/input/requirements/tomatinare/tomatinare"  # your input dataset
dst = "/kaggle/working/tomato"              # working folder

# Copy all files to working
shutil.copytree(src, dst)


In [ ]:
!pip show tensorflow
!pip show keras
!python --version

In [ ]:
import os
import random
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
TARGET = 3500

DATA_DIR = "/kaggle/working/tomato"
BALANCED_DIR = "/kaggle/working/balanced_data"

os.makedirs(BALANCED_DIR, exist_ok=True)

# -------------------- AUGMENTER --------------------
augmenter = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True
)

random.seed(SEED)

for cls in sorted(os.listdir(DATA_DIR)):
    print(f"\nProcessing class: {cls}")

    src = os.path.join(DATA_DIR, cls)
    dst = os.path.join(BALANCED_DIR, cls)
    os.makedirs(dst, exist_ok=True)

    images = os.listdir(src)

    # ---------- 1. COPY ORIGINAL IMAGES ----------
    for img in images:
        tf.io.gfile.copy(
            os.path.join(src, img),
            os.path.join(dst, img),
            overwrite=True
        )

    count = len(images)

    # ---------- 2. REMOVE EXTRA ----------
    if count > TARGET:
        extra = count - TARGET
        remove_imgs = random.sample(os.listdir(dst), extra)
        print(f"Deleting {extra} images")

        for img in remove_imgs:
            os.remove(os.path.join(dst, img))

    # ---------- 3. AUGMENT MISSING ----------
    elif count < TARGET:
        needed = TARGET - count
        print(f"Augmenting {needed} images")

        aug_gen = augmenter.flow_from_directory(
            BALANCED_DIR,
            classes=[cls],
            target_size=(128,128),
            batch_size=1,
            save_to_dir=dst,
            save_prefix="aug",
            save_format="jpg",
            class_mode=None,
            seed=SEED
        )

        for _ in range(needed):
            next(aug_gen)

print("\n✅ Dataset balanced successfully")


In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

SEED = 42
SPLIT = [0.7, 0.15, 0.15]

SRC_DIR = "/kaggle/working/balanced_data"
DST_DIR = "/kaggle/working/balanced_split"

splits = ["train", "val", "test"]

# Create split folders
for split in splits:
    os.makedirs(os.path.join(DST_DIR, split), exist_ok=True)

# Loop through balanced classes
for class_name in sorted(os.listdir(SRC_DIR)):
    class_path = os.path.join(SRC_DIR, class_name)

    if not os.path.isdir(class_path):
        continue

    images = [f for f in os.listdir(class_path)
              if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    if len(images) == 0:
        continue

    # Split
    train_imgs, temp_imgs = train_test_split(
        images,
        test_size=(1 - SPLIT[0]),
        random_state=SEED,
        shuffle=True
    )

    val_imgs, test_imgs = train_test_split(
        temp_imgs,
        test_size=SPLIT[2] / (SPLIT[1] + SPLIT[2]),
        random_state=SEED,
        shuffle=True
    )

    # Copy files
    for split_name, split_imgs in zip(splits, [train_imgs, val_imgs, test_imgs]):
        split_class_dir = os.path.join(DST_DIR, split_name, class_name)
        os.makedirs(split_class_dir, exist_ok=True)

        for img in split_imgs:
            src = os.path.join(class_path, img)
            dst = os.path.join(split_class_dir, img)

            if os.path.exists(src):
                shutil.copy(src, dst)

    print(f"✔ {class_name}: "
          f"train={len(train_imgs)}, "
          f"val={len(val_imgs)}, "
          f"test={len(test_imgs)}")

print("\n✅ Balanced dataset split completed")


In [ ]:
import os

BALANCED_DIR = "/kaggle/working/balanced_data"

for cls in os.listdir(BALANCED_DIR):
    cls_path = os.path.join(BALANCED_DIR, cls)
    if os.path.isdir(cls_path):
        print(cls, "->", len(os.listdir(cls_path)))


In [ ]:
SPLIT_DIR = "/kaggle/working/balanced_split"

for folder in ["train", "val", "test"]:
    path = os.path.join(SPLIT_DIR, folder)
    print("\n", folder.upper())

    for cls in os.listdir(path):
        cls_path = os.path.join(path, cls)
        if os.path.isdir(cls_path):
            print(cls, "->", len(os.listdir(cls_path)))


In [ ]:
dataset_path = "/kaggle/working/balanced_split"

for folder in ['train', 'val', 'test']:
    path = os.path.join(dataset_path, folder)
    print(folder, ":", sum(len(files) for _, _, files in os.walk(path)))



In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
IMG_SIZE = (128,128)
BATCH = 32

BASE_PATH = "/kaggle/working/balanced_split"

# ---------------- TRAIN ----------------
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    brightness_range=[0.7,1.3],
    horizontal_flip=True,
    vertical_flip=True
).flow_from_directory(
    f"{BASE_PATH}/train",
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode="categorical",
    shuffle=True,
    seed=SEED
)

# ---------------- VALIDATION ----------------
val_gen = ImageDataGenerator(
    rescale=1./255
).flow_from_directory(
    f"{BASE_PATH}/val",
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode="categorical",
    shuffle=False
)

# ---------------- TEST ----------------
test_gen = ImageDataGenerator(
    rescale=1./255
).flow_from_directory(
    f"{BASE_PATH}/test",
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode="categorical",
    shuffle=False
)


In [ ]:
from tensorflow.keras import layers, models

def cnn5():
    model = models.Sequential([
        # Input layer updated to 128x128
        layers.Conv2D(32, 3, activation='relu', padding='same', input_shape=(128,128,3)),
        layers.MaxPool2D(),

        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPool2D(),

        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPool2D(),

        layers.Conv2D(256, 3, activation='relu', padding='same'),
        layers.MaxPool2D(),

        layers.Conv2D(256, 3, activation='relu', padding='same'),
        layers.GlobalAveragePooling2D(),

        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(10, activation='softmax')  # 10 classes
    ])

    model.compile(optimizer='adam', 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])
    return model

# Instantiate and view model summary
model5 = cnn5()
model5.summary()


In [ ]:
!python --version

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger


checkpoint = ModelCheckpoint(
    filepath="/kaggle/working/best_model_60ep.h5",
    monitor="val_accuracy",
    save_best_only=True,
    save_weights_only=False,   # keep full model → smaller than saving weights separately
    mode="max",
    verbose=1
)

csv_logger = CSVLogger(
    "/kaggle/working/log_60ep.csv",
    append=False
)


history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=60,
    callbacks=[checkpoint, csv_logger],
    verbose=1
)

print("🔥 Training Completed Successfully!")
print("Files saved in working directory:")
print(" - best_model_60ep.h5")
print(" - log_60ep.csv")


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Test data must NOT be augmented
test_datagen = ImageDataGenerator(rescale=1.0/255.0)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

# ===========================
# 1️⃣ LOAD MODEL + CSV LOG
# ===========================

model_path = "/kaggle/working/best_model_60ep.h5"
csv_path   = "/kaggle/working/log_60ep.csv"

model = load_model(model_path)
history_df = pd.read_csv(csv_path)

print("Model and training log loaded successfully!")

# ===========================
# 2️⃣ BUILD TEST GENERATOR
# ===========================

test_data = test_datagen.flow_from_directory(
    directory="/kaggle/working/tomato/PlantVillage/test",
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# ===========================
# 3️⃣ RUN PREDICTIONS
# ===========================

pred_probs = model.predict(test_data)
y_pred = np.argmax(pred_probs, axis=1)
y_true = test_data.classes
class_labels = list(test_data.class_indices.keys())

# ===========================
# 4️⃣ CONFUSION MATRIX
# ===========================

cm = confusion_matrix(y_true, y_pred)

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.figure(figsize=(12, 9))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',          # clean, modern
    linewidths=0.8,        # thin grid lines
    linecolor='black',
    cbar=True,
    xticklabels=class_labels,
    yticklabels=class_labels,
    annot_kws={"size": 12, "weight": "bold"}   # bold numbers
)

plt.title("Confusion Matrix", fontsize=20, weight='bold')
plt.xlabel("Predicted Label", fontsize=16)
plt.ylabel("Actual Label", fontsize=16)

plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(rotation=0, fontsize=12)

plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=300)
plt.show()

print("Confusion matrix saved: /kaggle/working/confusion_matrix.png")

# ===========================
# 5️⃣ CLASSIFICATION REPORT
# ===========================

report = classification_report(
    y_true, y_pred, target_names=class_labels, output_dict=False
)
print("Classification Report:\n")
print(report)

# ===========================
# 6️⃣ INDIVIDUAL METRICS
# ===========================

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average="macro")
rec  = recall_score(y_true, y_pred, average="macro")
f1   = f1_score(y_true, y_pred, average="macro")

print("\n📊 **Evaluation Metrics**")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")

# ===========================
# 7️⃣ TRAINING CURVES
# ===========================

# Accuracy Curve
plt.figure(figsize=(8,5))
plt.plot(history_df["accuracy"], label="Train Accuracy")
plt.plot(history_df["val_accuracy"], label="Val Accuracy")
plt.title("Accuracy Curve")
plt.legend()
plt.savefig("/kaggle/working/accuracy_curve.png")
plt.show()

# Loss Curve
plt.figure(figsize=(8,5))
plt.plot(history_df["loss"], label="Train Loss")
plt.plot(history_df["val_loss"], label="Val Loss")
plt.title("Loss Curve")
plt.legend()
plt.savefig("/kaggle/working/loss_curve.png")
plt.show()

print("\nSaved:")
print(" - /kaggle/working/accuracy_curve.png")
print(" - /kaggle/working/loss_curve.png")


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D,
    BatchNormalization, Dropout,
    Dense, GlobalAveragePooling2D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

num_classes = train_gen.num_classes

modelopt = Sequential([
    Input(shape=(128,128,3)),

    # ================= BLOCK 1 (2 convs) =================
    Conv2D(32, (3,3), padding='same', activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Conv2D(32, (3,3), padding='same', activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    MaxPooling2D(),
    Dropout(0.20),

    # ================= BLOCK 2 (2 convs) =================
    Conv2D(64, (3,3), padding='same', activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Conv2D(64, (3,3), padding='same', activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    MaxPooling2D(),
    Dropout(0.25),

    # ================= BLOCK 3 (2 convs) =================
    Conv2D(128, (3,3), padding='same', activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Conv2D(128, (3,3), padding='same', activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    MaxPooling2D(),
    Dropout(0.30),

    # ================= BLOCK 4 (2 convs) =================
    Conv2D(256, (3,3), padding='same', activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Conv2D(256, (3,3), padding='same', activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    MaxPooling2D(),
    Dropout(0.35),

    # ================= BLOCK 5 (2 convs) =================
    Conv2D(512, (3,3), padding='same', activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Conv2D(512, (3,3), padding='same', activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    MaxPooling2D(),
    Dropout(0.40),

    # ================= HEAD =================
    GlobalAveragePooling2D(),

    Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.50),

    Dense(num_classes, activation='softmax')
])

modelopt.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

modelopt.summary()


In [ ]:
import tensorflow as tf

tf.config.run_functions_eagerly(True)


In [ ]:
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    ReduceLROnPlateau
)
import os

OUTPUT_DIR = "/kaggle/working/demo"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- CHECKPOINT ----------------
ckpt10 = ModelCheckpoint(
    filepath=os.path.join(OUTPUT_DIR, "demo.keras"),
    monitor="val_accuracy",
    save_best_only=True,
    save_weights_only=False,   # IMPORTANT
    verbose=1
)


# ---------------- LR SCHEDULER ----------------
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=5,
    min_lr=1e-6,
    verbose=1
)


In [ ]:
!pip show tensorflow
!pip show keras
!python --version

In [ ]:
history = modelopt.fit(
    train_gen,
    validation_data=val_gen,
    epochs=100,
    callbacks=[ckpt10,reduce_lr]
)


In [ ]:
history = modelopt.fit(
    train_gen,
    validation_data=val_gen,
    epochs=2,
    callbacks=[ckpt10,reduce_lr]
)


In [ ]:
from tensorflow.keras.models import load_model

model = load_model("/kaggle/working/demo/demo.keras", compile=False)
model.summary()


In [ ]:
# After training
model.save("modeldemo.h5")  # HDF5 format


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, f1_score, precision_score, recall_score,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
test_gen.reset()

X_test = []
y_test = []

for i in range(len(test_gen)):
    x, y = test_gen[i]
    X_test.append(x)
    y_test.append(y)

X_test = np.vstack(X_test)
y_test = np.argmax(np.vstack(y_test), axis=1)

class_names = list(test_gen.class_indices.keys())
num_classes = len(class_names)

print("Test samples:", X_test.shape[0])
print("Classes:", class_names)
y_pred_probs = modelopt.predict(X_test, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10,8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="macro")
rec = recall_score(y_test, y_pred, average="macro")
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}\n")

print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=class_names))
plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Val Accuracy")
plt.title("Accuracy Curve")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("Loss Curve")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.show()
f1_scores = f1_score(y_test, y_pred, average=None)
prec_scores = precision_score(y_test, y_pred, average=None)
rec_scores = recall_score(y_test, y_pred, average=None)

x = np.arange(num_classes)

plt.figure(figsize=(12,6))
plt.bar(x - 0.25, f1_scores, 0.25, label="F1-score")
plt.bar(x, prec_scores, 0.25, label="Precision")
plt.bar(x + 0.25, rec_scores, 0.25, label="Recall")

plt.xticks(x, class_names, rotation=45)
plt.ylabel("Score")
plt.title("Class-wise Performance Metrics")
plt.legend()
plt.tight_layout()
plt.show()
y_test_bin = label_binarize(y_test, classes=range(num_classes))

plt.figure(figsize=(10,8))
for i in range(num_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_names[i]} (AUC={roc_auc:.2f})")

plt.plot([0,1], [0,1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.show()
plt.figure(figsize=(10,8))
for i in range(num_classes):
    p, r, _ = precision_recall_curve(y_test_bin[:, i], y_pred_probs[:, i])
    ap = average_precision_score(y_test_bin[:, i], y_pred_probs[:, i])
    plt.plot(r, p, label=f"{class_names[i]} (AP={ap:.2f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curves")
plt.legend()
plt.show()
mis_idx = np.where(y_pred != y_test)[0]

plt.figure(figsize=(12,12))
for i, idx in enumerate(mis_idx[:16]):
    plt.subplot(4,4,i+1)
    plt.imshow(X_test[idx])
    plt.title(
        f"T: {class_names[y_test[idx]]}\nP: {class_names[y_pred[idx]]}",
        fontsize=9
    )
    plt.axis("off")

plt.suptitle("Misclassified Images")
plt.tight_layout()
plt.show()
if "lr" in history.history:
    plt.plot(history.history["lr"])
    plt.title("Learning Rate Schedule")
    plt.xlabel("Epochs")
    plt.ylabel("Learning Rate")
    plt.show()


In [ ]:
!pip show tensorflow
!pip show keras

In [ ]:
import os
import shutil

# Paths to files and folders you want to combine
paths_to_collect = [
    "/kaggle/working/model100_best_comic.keras",
    "/kaggle/working/model100_new_best.weights.h5",
    "/kaggle/working/models/",
    "/kaggle/working/results/"
]

# Path for the new combined folder
combined_folder = "/kaggle/working/all_models_results"
os.makedirs(combined_folder, exist_ok=True)

# Move all files/folders into the combined folder
for path in paths_to_collect:
    if os.path.exists(path):
        # If it's a folder, move its contents
        if os.path.isdir(path):
            for item in os.listdir(path):
                src_path = os.path.join(path, item)
                dest_path = os.path.join(combined_folder, item)
                shutil.move(src_path, dest_path)
       
        else:
            # If it's a single file
            shutil.move(path, os.path.join(combined_folder, os.path.basename(path)))

# Create a zip of the combined folder
shutil.make_archive("/kaggle/working/all_models_results", 'zip', combined_folder)

print("All models and results are collected and zipped at /kaggle/working/all_models_results.zip")


In [ ]:
# ================================
# 1️⃣ Import Libraries
# ================================
import numpy as np
from tensorflow.keras.models import load_model
from PIL import Image
import io
import ipywidgets as widgets
from IPython.display import display

# ================================
# 2️⃣ Load your .keras model
# ================================
MODEL_PATH = "/kaggle/input/tomato/tensorflow2/default/1/model100_best.keras"  # <-- Replace with your .keras path
model = load_model(MODEL_PATH, compile=False)
model.summary()

# ================================
# 3️⃣ Define your class names
# ================================
classes = [
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites_Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy"
]  # <-- Replace with your actual classes if different

# ================================
# 4️⃣ Define prediction function
# ================================
def predict(change):
    if not uploader.value:
        return
    
    uploaded_file = list(uploader.value.values())[0]
    content = uploaded_file['content']
    
    # Open image and resize
    img = Image.open(io.BytesIO(content)).convert('RGB')
    img = img.resize((128, 128))  # Must match model input size
    x = np.array(img) / 255.0
    x = np.expand_dims(x, axis=0)
    
    # Make prediction
    preds = model.predict(x)
    class_idx = np.argmax(preds)
    probability = np.max(preds)
    
    print(f"Predicted Class: {classes[class_idx]}")
    print(f"Probability: {probability:.4f}")

# ================================
# 5️⃣ File uploader widget
# ================================
uploader = widgets.FileUpload(
    accept='image/*',
    multiple=False
)
uploader.observe(predict, names='value')
display(uploader)


In [ ]:
# ================================
# 1️⃣ Import Libraries
# ================================
import numpy as np
from tensorflow.keras.models import load_model
from PIL import Image
import os

# ================================
# 2️⃣ Load your .keras model
# ================================
MODEL_PATH = "/kaggle/input/tomato/tensorflow2/default/1/model100_best.keras"  # <-- Replace with your uploaded model path
model = load_model(MODEL_PATH, compile=False)
model.summary()

# ================================
# 3️⃣ Define your class names
# ================================
classes = [
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites_Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy"
]  # Replace with your classes

# ================================
# 4️⃣ Upload Image
# ================================
from IPython.display import display
from ipywidgets import FileUpload

uploader = FileUpload(accept='image/*', multiple=False)
display(uploader)


In [ ]:
import io

# ================================
# 5️⃣ Predict uploaded image
# ================================
if len(uploader.value) > 0:
    # For Kaggle, uploader.value is a tuple
    uploaded_file = uploader.value[0]  
    content = uploaded_file['content']
    
    # Open image, resize and normalize
    img = Image.open(io.BytesIO(content)).convert('RGB')
    img = img.resize((128,128))
    x = np.array(img)/255.0
    x = np.expand_dims(x, axis=0)
    
    # Predict
    preds = model.predict(x)
    class_idx = np.argmax(preds)
    probability = np.max(preds)
    
    print(f"Predicted Class: {classes[class_idx]}")
    print(f"Probability: {probability:.4f}")
else:
    print("Upload an image first!")
    


In [ ]:
from tensorflow.keras.models import load_model

model = load_model("/kaggle/input/tomato/tensorflow2/default/1/model100_best.keras")
model.save("model.h5")


In [11]:
import zipfile

h5_file = "model.h5"
zip_file = "model.zip"

with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(h5_file)

print("ZIP file created successfully!")


ZIP file created successfully!
